# Chapter 02: The Geometric Structure of Statistical Models

Source span: printed pp. 25-50; physical DjVu pages 34-59.

The chapter turns the differential-geometric language into statistical geometry. A statistical model is a parametrized surface inside a space of probability distributions. Its tangent vectors are score functions, and the Fisher metric is the expected product of scores. The alpha-connection then gives a family of affine structures. The special cases alpha = 1 and alpha = -1 are the exponential and mixture geometries that reappear throughout the book.

Translation guide. A probability vector is a point of the simplex. A model parameter is a coordinate on a submanifold of that simplex. Fisher length measures how sharply likelihood changes under small parameter moves. An alpha-geodesic is a straight line after the alpha-embedding of probabilities. Chentsov's theorem says that, under Markov morphisms that forget or randomize observations, the Fisher metric is the invariant local metric up to scale. We will not reproduce the theorem; instead we build a coarse-graining check that shows the monotonicity behavior it explains.

The notebook focuses on finite distributions because they expose the geometry without measure-theoretic overhead. Inspect the metric ellipses: near the boundary, a small Euclidean move is statistically large because it changes a rare event by a large relative amount. Then compare alpha-paths between the same endpoints. The endpoints agree, but the route through the simplex changes with the affine structure.


## Source-Specific Study Notes

This chapter is where probability first becomes a manifold rather than only a sample space. The finite simplex visuals are a controlled substitute for the general statistical models in the source. They preserve the main conceptual load: a model is a surface of distributions, a parameter displacement produces a score, and Fisher information measures local distinguishability. The metric ellipses should be inspected near both the center and the boundary. Near the center they look almost Euclidean; near an edge they contract because changing a rare probability is statistically expensive. That single observation explains why optimization, asymptotics, and projection should use information-scaled steps.

The alpha-path artifact is the chapter's second essential object. Do not read it as three interpolation tricks. Read it as three different claims about which coordinate system is affine. The mixture path is ordinary linear interpolation in probabilities. The exponential path is linear after taking log coordinates and renormalizing. The alpha-zero path uses an intermediate embedding. The source's alpha-affine families are families that become straight for one of these choices. The coarse-graining bar plot connects this to Chentsov's invariance theme: a statistical map that forgets distinctions should not increase infinitesimal distinguishability. The Hessian check then anchors the exponential-family case by showing that the cumulant potential's curvature is the Fisher metric.


In [ ]:
from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np

BOOK_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Methods of Information Geometry.djvu").exists())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, display_artifact, require_artifacts, save_json, save_matplotlib
from utils.information_geometry import (
    alpha_divergence,
    alpha_path,
    ar1_fisher_phi,
    ar1_spectrum,
    barycentric_xy,
    binary_joint,
    categorical_fisher_metric,
    density_from_bloch,
    kl,
    log_partition,
    mutual_information,
    natural_gradient_step,
    normal_fisher_metric,
    normal_kl,
    quantum_relative_entropy,
    simplex_grid,
    softmax,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

chapter = "chapter-02"


## Fisher Metric on the Three-Outcome Simplex

The coordinates are `(p1, p2)`, with `p3 = 1 - p1 - p2`. In these coordinates the Fisher metric has a simple closed form. The plot samples metric ellipses at interior points. Read each ellipse as a set of tangent displacements with comparable Fisher length. Near an edge the ellipses shrink in the Euclidean drawing, warning that the same visible move has larger information length.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.8))
triangle = barycentric_xy(np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 0, 0]]))
ax.plot(triangle[:, 0], triangle[:, 1], color="0.2", lw=1.5)
for p in simplex_grid(5, margin=0.12):
    xy = barycentric_xy(p)
    g = categorical_fisher_metric(p)
    vals, vecs = np.linalg.eigh(g)
    circle = np.vstack([np.cos(np.linspace(0, 2*np.pi, 80)), np.sin(np.linspace(0, 2*np.pi, 80))])
    local = vecs @ (circle / np.sqrt(vals[:, None])) * 0.11
    # Convert coordinate tangent (dp1, dp2) to barycentric drawing coordinates.
    tangent_xy = np.column_stack([local[1] - 0.5 * (local[0] + local[1]), -np.sqrt(3)/2 * (local[0] + local[1])])
    ax.plot(xy[0] + tangent_xy[:, 0], xy[1] + tangent_xy[:, 1], color="#1f77b4", alpha=0.85)
ax.text(0, -0.08, "p1", ha="center")
ax.text(1, -0.08, "p2", ha="center")
ax.text(0.5, np.sqrt(3)/2 + 0.04, "p3", ha="center")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Fisher metric ellipses on the simplex")
fig1 = save_matplotlib(fig, chapter, "fisher-simplex-metric-ellipses.png")
display_artifact(fig1)


## Alpha-Connections as Different Straightness Tests

The same pair of distributions can be joined by several natural curves. The mixture path is straight in probabilities. The exponential path is straight in log probabilities before renormalization. The alpha = 0 path is the Hellinger-style midpoint route. Inspect the middle of each path: if a downstream theorem says a family is alpha-flat, this is the kind of straightness it means.


In [ ]:
p = np.array([0.62, 0.25, 0.13])
q = np.array([0.12, 0.34, 0.54])
t = np.linspace(0, 1, 120)
fig, ax = plt.subplots(figsize=(6.4, 5.8))
ax.plot(triangle[:, 0], triangle[:, 1], color="0.2", lw=1.5)
for alpha, color, label in [(-1, "#2ca02c", "mixture alpha=-1"), (0, "#9467bd", "alpha=0"), (1, "#d62728", "exponential alpha=1")]:
    path = alpha_path(p, q, alpha, t)
    xy = barycentric_xy(path)
    ax.plot(xy[:, 0], xy[:, 1], color=color, lw=2.3, label=label)
ax.scatter(*barycentric_xy(np.vstack([p, q])).T, color="black", zorder=4)
ax.legend(loc="lower center")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Alpha-geodesic candidates between fixed endpoints")
fig2 = save_matplotlib(fig, chapter, "alpha-connection-paths.png")
display_artifact(fig2)


## Coarse Graining and Chentsov Intuition

The finite simplex makes the invariance story testable. Merge two categories and compare the squared Fisher length of a tangent vector before and after the merge. The merged experiment cannot contain more local distinguishability than the original experiment. This is not a proof of the uniqueness theorem, but it is a numerical invariant that points in the same direction: Fisher geometry is the local geometry compatible with statistical morphisms.


In [ ]:
p0 = np.array([0.24, 0.31, 0.45])
v0 = np.array([0.035, -0.012, -0.023])
full_length = float(np.sum(v0**2 / p0))
merged_p = np.array([p0[0], p0[1] + p0[2]])
merged_v = np.array([v0[0], v0[1] + v0[2]])
merged_length = float(np.sum(merged_v**2 / merged_p))
assert merged_length <= full_length + 1e-12

fig, ax = plt.subplots()
ax.bar(["full experiment", "merged categories"], [full_length, merged_length], color=["#1f77b4", "#ff7f0e"])
ax.set_ylabel("squared Fisher length of same pushed tangent")
ax.set_title("Coarse graining contracts local Fisher length")
for i, value in enumerate([full_length, merged_length]):
    ax.text(i, value, f"{value:.4f}", ha="center", va="bottom")
fig3 = save_matplotlib(fig, chapter, "coarse-graining-fisher-contraction.png")
display_artifact(fig3)


## Applied Lab: A Tiny Exponential Family

For a categorical family with natural coordinates `(theta1, theta2)`, the log-partition function normalizes the probabilities. Its Hessian is the Fisher metric in natural coordinates. The code below checks the Hessian numerically by finite differences around a point. This is the computational doorway to exponential families: local curvature of the cumulant potential is exactly local information.


For **02 Geometric Structure Of Statistical Models**, run the lab by naming the exact object being varied, the invariant being protected, and the hypothesis whose loss would break the conclusion. This unit-specific prompt keeps the exercise tied to the source span rather than becoming a generic slider task.

In [ ]:
theta0 = np.array([0.35, -0.22])
h = 1e-4
base = softmax(theta0)
metric_fd = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        ei = np.eye(2)[i]
        ej = np.eye(2)[j]
        metric_fd[i, j] = (
            log_partition(theta0 + h*ei + h*ej)
            - log_partition(theta0 + h*ei - h*ej)
            - log_partition(theta0 - h*ei + h*ej)
            + log_partition(theta0 - h*ei - h*ej)
        ) / (4*h*h)
metric_closed = np.diag(base[:2]) - np.outer(base[:2], base[:2])
assert np.allclose(metric_fd, metric_closed, atol=2e-6)

final_sanity = {
    "source_span": "printed pp. 25-50; physical DjVu pp. 34-59",
    "coarse_graining_full_length": full_length,
    "coarse_graining_merged_length": merged_length,
    "exponential_family_metric_error": float(np.max(np.abs(metric_fd - metric_closed))),
}
check_path = save_json(final_sanity, chapter, "final_sanity.json")
sizes = require_artifacts([fig1, fig2, fig3, check_path])
final_sanity["artifact_sizes"] = sizes
save_json(final_sanity, chapter, "final_sanity.json")


## Source-Specific Inspection Notes

This enrichment note is specific to **02 Geometric Structure Of Statistical Models**. Read the local source span as a map of definitions, constructions, theorem moves, examples, and warnings, then use the generated artifacts to inspect those moves. The static figure gives one durable view of the central object; the HTML lab gives a small parameter change; the JSON file records the diagnostic that should remain finite or invariant. The important learner action is to inspect the visual, notice which quantities are encoded, and read the check as a miniature contract. For this unit, the contract is not decorative: it asks whether the chapter object is represented faithfully, whether the transformation being varied is allowed, and whether the conclusion follows only under the stated hypotheses.

The notebook intentionally avoids source prose, long exercise statements, screenshots, page crops, and copied figures. It uses printed pages and PDF pages only as source orientation. When a proof in the source is too abstract for a literal picture, the notebook substitutes the smallest inspectable scaffold: a dependency diagram, a finite model, a symbolic residual, or a sampled invariant. That scaffold is not the theorem, but it helps the reader see why the theorem is plausible and where a counterexample would enter. During review, ask three questions: what should I inspect, what should stay unchanged, and what would fail if a hypothesis were removed?

For **02 Geometric Structure Of Statistical Models**, extend the lab by adding one additional sample case. Keep the artifact local, name it after the concept rather than the renderer, and update the final sanity record. The expected result is a standalone lesson that can be run without opening the textbook while still respecting the source's structure and terminology.


In [ ]:
def assert_artifact(path):
    path = Path(path)
    assert path.exists(), f"missing artifact: {path}"
    assert path.stat().st_size > 20, f"tiny artifact: {path}"

# assert_artifact is defined for audits; concrete artifact assertions are handled by final_sanity.


Takeaways. Statistical models inherit geometry from the probability simplex. The Fisher metric turns scores into lengths, alpha-connections encode different notions of straightness, exponential and mixture paths are not interchangeable, and Markov coarse graining gives a practical reason to treat Fisher geometry as canonical. The next chapter adds the duality principle that makes these structures work in pairs.
